# Collect eagle's calibration data on Kaggle

This notebook only calls `calibrate.sh` in `eagle/benchmarks/reproduce/` (see its README). It
times a few isolated micro-kernels on this machine and writes one CSV of raw timings; nothing is
fitted and nothing is changed. Before running:

- **Settings -> Accelerator**: GPU T4 (the run uses the first GPU).
- **Settings -> Internet**: on (the pinned pre-release wheels are downloaded).

The run takes a few minutes. Download the CSV from the notebook output when it finishes.

In [ ]:
# Parameters
WHICH = "gpu"           # gpu | cpu | all  (cpu measures the 4 shared vCPUs of the session)
QUICK = False           # True: a one-minute smoke pass
EAGLE_REF = "main"      # eagle ref to fetch the scripts from

In [ ]:
import os, pathlib, subprocess
WORK = pathlib.Path("/kaggle/temp" if pathlib.Path("/kaggle/temp").is_dir() else "/tmp") / "raptor-repro"
SRC, OUT = WORK / "src", pathlib.Path("/kaggle/working/calibration")
SRC.mkdir(parents=True, exist_ok=True)
if not (SRC / "eagle").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", EAGLE_REF,
                    "https://github.com/amasat01/eagle.git", str(SRC / "eagle")], check=True)
REPRO = SRC / "eagle" / "benchmarks" / "reproduce"
print(REPRO)

In [ ]:
# Install the pinned pre-release wheels, run the collector, print where the CSV went.
env = {**os.environ, "CUDA_VISIBLE_DEVICES": os.environ.get("CUDA_VISIBLE_DEVICES", "0")}
for prof in {"gpu": ["gpu"], "cpu": ["cpu"], "all": ["gpu", "cpu"]}[WHICH]:
    cmd = ["bash", str(REPRO / "calibrate.sh"), "--profile", prof, "--work", str(WORK),
           "--out", str(OUT / prof)] + (["--quick"] if QUICK else [])
    rc = subprocess.run(cmd, env=env).returncode
    print(prof, {0: "done"}.get(rc, f"failed (exit {rc}): see the output above"))

In [ ]:
# The head of each CSV (machine facts, then the first rows) and the wall of each group.
for p in sorted(OUT.glob("*/calibration-*.csv")):
    lines = p.read_text().splitlines()
    print(p)
    print("\n".join(ln for ln in lines[:40] if ln.startswith("# ") or ln.startswith("case_id")))
    print("\n".join(lines[len([ln for ln in lines if ln.startswith("#")]) + 1:][:6]))
    print()